In [ ]:
# %%
# Compare forward-only vs. backward (full) vs. backward (inputs only)
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

# --- Config ---
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

# --- Simple model ---
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1024, 2048),
            nn.ReLU(),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
        )
    def forward(self, x):
        return self.net(x)

model = TinyNet().to(device)

# Dummy input
x = torch.randn(256, 1024, device=device, requires_grad=True)

# Utility function
def time_it(fn, n=10, desc=""):
    torch.cuda.synchronize() if device == "cuda" else None
    t0 = time.time()
    for _ in range(n):
        fn()
    torch.cuda.synchronize() if device == "cuda" else None
    print(f"{desc:<40s} {1000*(time.time() - t0)/n:.3f} ms per run")

# 1. Forward-only (no gradients)
def forward_only():
    with torch.inference_mode():
        _ = model(x)

# 2. Full backward (inputs + weights)
def full_backward():
    x_ = x.clone().detach().requires_grad_(True)
    out = model(x_)
    loss = out.pow(2).mean()
    loss.backward()  # grads w.r.t. both inputs and params

# 3. Optimized backward: grads only for inputs (skip param grads)
def backward_inputs_only():
    # Temporarily disable param grads
    for p in model.parameters():
        p.requires_grad_(False)

    x_ = x.clone().detach().requires_grad_(True)
    out = model(x_)
    grad_inp = torch.autograd.grad(out.pow(2).mean(), x_)[0]

    # Re-enable param grads for fairness
    for p in model.parameters():
        p.requires_grad_(True)

# Warm-up (GPU)
for _ in range(5):
    forward_only(); full_backward(); backward_inputs_only()
torch.cuda.synchronize() if device == "cuda" else None

# --- Timing ---
print("=== Runtime Comparison ===")
time_it(forward_only, 20, "Forward only (inference_mode)")
time_it(full_backward, 20, "Backward (inputs + weights)")
time_it(backward_inputs_only, 20, "Backward (inputs only)")


In [118]:
model = torch.compile(model)

In [137]:
# %%
# Compare forward-only vs. backward (full) vs. backward (inputs only)
# vs. backward (inputs only, detached params) vs. compiled variants
import torch
import torch.nn as nn
import time, copy

# --- Config ---
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

# --- Simple model ---
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1024, 2048),
            nn.ReLU(),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
        )
    def forward(self, x):
        return self.net(x)

model = TinyNet().to(device)

# Dummy input
x = torch.randn(256, 1024, device=device, requires_grad=True)

# --- Utility ---
def time_it(fn, n=10, desc=""):
    torch.cuda.synchronize() if device == "cuda" else None
    t0 = time.time()
    for _ in range(n):
        fn()
    torch.cuda.synchronize() if device == "cuda" else None
    print(f"{desc:<50s} {1000*(time.time() - t0)/n:.3f} ms per run")

# --- Define test functions ---
def forward_only():
    with torch.inference_mode():
        _ = model(x)

def full_backward():
    x_ = x.clone().detach().requires_grad_(True)
    out = model(x_)
    loss = out.pow(2).mean()
    loss.backward()  # grads w.r.t. both inputs and params

def backward_inputs_only():
    x_ = x.clone().detach().requires_grad_(True)
    out = model(x_)
    grad_inp = torch.autograd.grad(out.pow(2).mean(), x_, retain_graph=False)[0]
    _ = grad_inp.norm()

def backward_inputs_only_detached():
    x_ = x.clone().detach().requires_grad_(True)
    out = model(x_)
    out.pow(2).mean().backward()

# --- Compile variants ---
# Copy model for frozen params before compiling
model_inputs_only = copy.deepcopy(model)
for p in model_inputs_only.parameters():
    p.requires_grad_(False)

try:
    model_train_compiled = torch.compile(copy.deepcopy(model))   # full grads
    model_inputs_only_compiled = torch.compile(model_inputs_only) # frozen params
    compiled_available = True
except Exception as e:
    print("torch.compile not available or failed:", e)
    compiled_available = False

def full_backward_compiled():
    x_ = x.clone().detach().requires_grad_(True)
    out = model_train_compiled(x_)
    loss = out.pow(2).mean()
    loss.backward()

def backward_inputs_only_compiled():
    x_ = x.clone().detach().requires_grad_(True)
    out = model_inputs_only_compiled(x_)
    grad_inp = torch.autograd.grad(out.pow(2).mean(), x_, retain_graph=False)[0]
    _ = grad_inp.norm()

# --- Warm-up (GPU)
for _ in range(5):
    forward_only()
    full_backward()
    backward_inputs_only()
    backward_inputs_only_detached()
    if compiled_available:
        full_backward_compiled()
        backward_inputs_only_compiled()
torch.cuda.synchronize() if device == "cuda" else None

# --- Timing ---



print("=== Runtime Comparison (short run, 200 iters) ===")
time_it(forward_only, 200, "Forward only (inference_mode)")
time_it(full_backward, 200, "Backward (inputs + weights)")

for p in model.parameters():
    p.requires_grad_(False)
time_it(backward_inputs_only, 200, "Backward (inputs only, no param grads)")
time_it(backward_inputs_only_detached, 200, "Backward (inputs only, detached params)")

if compiled_available:
    time_it(full_backward_compiled, 200, "Backward (compiled, full)")
    time_it(backward_inputs_only_compiled, 200, "Backward (compiled, inputs only)")

# Restore grad state
for p in model.parameters():
    p.requires_grad_(True)



# --- Long run ---
print("\n=== Runtime Comparison (long run, 5000 iters) ===")
time_it(forward_only, 5000, "Forward only (inference_mode)")
time_it(full_backward, 5000, "Backward (inputs + weights)")


for p in model.parameters():
    p.requires_grad_(False)
time_it(backward_inputs_only_detached, 5000, "Backward (inputs only, detached params)")
time_it(backward_inputs_only, 5000, "Backward (inputs only, no param grads)")


if compiled_available:
    time_it(full_backward_compiled, 5000, "Backward (compiled, full)")
    time_it(backward_inputs_only_compiled, 5000, "Backward (compiled, inputs only)")

for p in model.parameters():
    p.requires_grad_(True)


=== Runtime Comparison (short run, 200 iters) ===
Forward only (inference_mode)                      0.563 ms per run
Backward (inputs + weights)                        1.395 ms per run
Backward (inputs only, no param grads)             0.650 ms per run
Backward (inputs only, detached params)            0.895 ms per run
Backward (compiled, full)                          0.995 ms per run
Backward (compiled, inputs only)                   0.810 ms per run

=== Runtime Comparison (long run, 5000 iters) ===
Forward only (inference_mode)                      0.217 ms per run
Backward (inputs + weights)                        0.774 ms per run
Backward (inputs only, detached params)            0.499 ms per run
Backward (inputs only, no param grads)             0.553 ms per run
Backward (compiled, full)                          1.016 ms per run
Backward (compiled, inputs only)                   0.790 ms per run


In [144]:
# %%
import torch
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

# --- Data ---
N, D = 50000, 512
Q_N = 256
K = 10  # top-k

X = torch.randn(N, D, device=device)
Q = torch.randn(Q_N, D, device=device, requires_grad=True)

# --- Custom autograd function for top-k distances ---
class TopKDistances(torch.autograd.Function):
    @staticmethod
    def forward(ctx, Q, X, k):
        """
        Compute all distances, select top-k, save only top-k info for backward.

        Args:
            Q: [Q_N, D] query vectors
            X: [N, D] database vectors
            k: number of nearest neighbors

        Returns:
            D_topk: [Q_N, k] top-k distances
        """
        # Compute all distances to find top-k (no grad needed here)
        with torch.no_grad():
            D_all = torch.cdist(Q, X)  # [Q_N, N]
            vals, idx = D_all.topk(k, dim=1, largest=False)  # [Q_N, k]

        # Save only top-k indices and the selected X vectors for backward
        X_topk = X[idx]  # [Q_N, k, D]
        ctx.save_for_backward(Q, X_topk, vals)
        ctx.k = k

        return vals

    @staticmethod
    def backward(ctx, grad_output):
        """
        Compute gradient w.r.t. Q using only saved top-k information.

        Args:
            grad_output: [Q_N, k] gradient from upstream loss

        Returns:
            grad_Q: [Q_N, D] gradient w.r.t. Q
            None: no gradient w.r.t. X
            None: no gradient w.r.t. k
        """
        Q, X_topk, D_topk = ctx.saved_tensors
        # Q: [Q_N, D]
        # X_topk: [Q_N, k, D]
        # D_topk: [Q_N, k]
        # grad_output: [Q_N, k]

        Q_expand = Q[:, None, :]  # [Q_N, 1, D]
        diff = Q_expand - X_topk  # [Q_N, k, D]

        # Gradient of L2 distance: d(||q-x||)/dq = (q-x)/||q-x||
        # Handle potential division by zero
        D_topk_safe = D_topk.clamp(min=1e-8)[:, :, None]  # [Q_N, k, 1]
        grad_dist = diff / D_topk_safe  # [Q_N, k, D]

        # Chain rule: multiply by upstream gradient
        grad_Q = (grad_output[:, :, None] * grad_dist).sum(dim=1)  # [Q_N, D]

        return grad_Q, None, None

# --- Timing utility ---
def time_it(fn, n=3, desc=""):
    torch.cuda.synchronize() if device == "cuda" else None
    t0 = time.time()
    for _ in range(n):
        fn()
    torch.cuda.synchronize() if device == "cuda" else None
    print(f"{desc:<60s} {1000*(time.time()-t0)/n:.3f} ms per run")

# --- 1. Forward-only ---
def forward_only():
    with torch.inference_mode():
        _ = torch.cdist(Q, X)

# --- 2. Full backward (inputs only) ---
def full_backward_inputs_only():
    Q_ = Q.clone().detach().requires_grad_(True)
    D = torch.cdist(Q_, X)           # forward
    loss = D.pow(2).mean()
    loss.backward()                   # backward is called automatically here
    _ = Q_.grad.norm()               # access the gradient

# --- 3. Top-k recompute backward (inputs only) ---
def topk_recompute_backward_inputs_only():
    Q_ = Q.clone().detach().requires_grad_(True)
    with torch.no_grad():
        D = torch.cdist(Q_, X)           # forward to select top-k (no grad)
        vals, idx = D.topk(K, dim=1, largest=False)
    # recompute distances only for top-k for backward
    X_topk = X[idx]                  # [Q_N, K, D]
    Q_expand = Q_[:, None, :].expand_as(X_topk)
    D_topk = (Q_expand - X_topk).pow(2).sum(dim=2).sqrt()
    loss = D_topk.pow(2).mean()
    loss.backward()
    _ = Q_.grad.norm()

# --- 4. Custom autograd function (save top-k, no recompute) ---
def custom_topk_backward_inputs_only():
    Q_ = Q.clone().detach().requires_grad_(True)
    D_topk = TopKDistances.apply(Q_, X, K)  # [Q_N, K]
    loss = D_topk.pow(2).mean()
    loss.backward()                  # backward is called here, triggers our custom backward
    _ = Q_.grad.norm()

# --- Warm-up ---
for _ in range(2):
    forward_only()
    full_backward_inputs_only()
    topk_recompute_backward_inputs_only()
    custom_topk_backward_inputs_only()
torch.cuda.synchronize() if device == "cuda" else None

# --- Timing ---
print("=== Euclidean distances: forward + backward (inputs only) ===")
time_it(forward_only, 300, "Forward-only (no grad)")
time_it(full_backward_inputs_only, 300, "Full cdist backward (inputs only)")
time_it(topk_recompute_backward_inputs_only, 300, "Top-k recompute backward (inputs only)")
time_it(custom_topk_backward_inputs_only, 300, "Custom autograd top-k (no recompute)")

# --- Verify gradient correctness ---
print("\n=== Gradient verification ===")

# Full backward
Q_test1 = Q.clone().detach().requires_grad_(True)
D = torch.cdist(Q_test1, X)
vals_full, idx = D.topk(K, dim=1, largest=False)
loss1 = vals_full.pow(2).mean()
loss1.backward()
grad1 = Q_test1.grad.clone()
print(f"Full backward grad: shape={grad1.shape}, norm={grad1.norm().item():.6f}")

# Custom autograd
Q_test2 = Q.clone().detach().requires_grad_(True)
D_topk = TopKDistances.apply(Q_test2, X, K)
loss2 = D_topk.pow(2).mean()
loss2.backward()
grad2 = Q_test2.grad.clone()
print(f"Custom backward grad: shape={grad2.shape}, norm={grad2.norm().item():.6f}")

# Check if gradients match
grad_diff = (grad1 - grad2).abs().max().item()
print(f"Max gradient difference: {grad_diff:.2e}")
print(f"Gradients match: {grad_diff < 1e-4}")

=== Euclidean distances: forward + backward (inputs only) ===
Forward-only (no grad)                                       2.508 ms per run
Full cdist backward (inputs only)                            7.209 ms per run
Top-k recompute backward (inputs only)                       3.407 ms per run
Custom autograd top-k (no recompute)                         3.359 ms per run

=== Gradient verification ===
Full backward grad: shape=torch.Size([256, 512]), norm=2.585506
Custom backward grad: shape=torch.Size([256, 512]), norm=2.585506
Max gradient difference: 5.59e-09
Gradients match: True
